In [31]:
import json
import os
import random
import numpy as np
from PIL import Image, ImageDraw
import glob
import datetime

def parse_drawing_data(drawing_data):
    """解析绘图数据，返回一系列笔画"""
    strokes = []
    for stroke in drawing_data:
        # 简化后的数据只有 x 和 y 坐标（没有时间信息）
        if len(stroke) >= 2:  # 确保至少有 x 和 y 坐标
            x_coords = stroke[0]
            y_coords = stroke[1]
            # 将 x 和 y 坐标配对
            points = list(zip(x_coords, y_coords))
            strokes.append(points)
    return strokes

def render_png(strokes, output_path, canvas_size=256, line_width=3):
    """将笔画渲染为 PNG 图像"""
    # 创建一个白色背景的图像
    image = Image.new('RGB', (canvas_size, canvas_size), color='white')
    draw = ImageDraw.Draw(image)

    # 绘制每个笔画
    for points in strokes:
        if len(points) > 1:  # 确保至少有两个点可以连接
            # 绘制线条
            draw.line(points, fill='black', width=line_width)

    # 保存图像
    image.save(output_path)
    return image

def count_lines(file_path):
    """计算文件中的行数"""
    with open(file_path, 'r') as f:
        return sum(1 for _ in f)

def get_random_line(file_path, line_count=None):
    """从文件中随机获取一行"""
    if line_count is None:
        line_count = count_lines(file_path)

    random_line_number = random.randint(1, line_count)

    with open(file_path, 'r') as f:
        for i, line in enumerate(f, 1):
            if i == random_line_number:
                return line.strip()

    return None  # 如果出现问题

def generate_random_quiz(ndjson_folder, output_folder, question_index):
    """
    从指定文件夹中随机选择一个NDJSON文件，
    从中随机抽取一行，生成图像作为测验题目
    """
    # 确保输出文件夹存在
    os.makedirs(output_folder, exist_ok=True)

    # 获取所有NDJSON文件
    ndjson_files = glob.glob(os.path.join(ndjson_folder, "*.ndjson"))

    if not ndjson_files:
        print(f"错误：在 {ndjson_folder} 中没有找到NDJSON文件")
        return None

    # 随机选择一个文件
    random_file = random.choice(ndjson_files)
    file_name = os.path.basename(random_file)
    category = os.path.splitext(file_name)[0]  # 不带扩展名的文件名作为类别

    print(f"已选择文件: {file_name}")

    # 从文件中随机选择一行
    random_line = get_random_line(random_file)

    if not random_line:
        print(f"错误：无法从 {file_name} 中读取数据")
        return None

    try:
        # 解析JSON数据
        data = json.loads(random_line)
        drawing_data = data.get('drawing', [])
        key_id = data.get('key_id', 'unknown')
        word = data.get('word', 'unknown')

        # 使用序号命名文件
        output_filename = f"{question_index}.png"
        output_path = os.path.join(output_folder, output_filename)

        # 解析绘图数据并渲染图像
        strokes = parse_drawing_data(drawing_data)
        render_png(strokes, output_path)

        # 创建答案文件
        answer_file = os.path.join(output_folder, f"{question_index}.txt")
        with open(answer_file, 'w') as f:
            f.write(f"{word}\n")

        print(f"已生成测验题目: {output_path}")
        print(f"答案已保存至: {answer_file}")

        quiz_info = {
            "image_path": output_path,
            "answer": category,
            "word": word,
            "key_id": key_id,
            "source_file": file_name
        }

        return quiz_info

    except json.JSONDecodeError:
        print(f"错误：无法解析JSON数据")
        return None
    except Exception as e:
        print(f"错误：{str(e)}")
        return None

def main():
    # Configure folder paths
    ndjson_folder = "ndjson_files"  # Folder containing NDJSON files
    output_folder = "quiz_images"   # Folder for output images and answers

    # Get user input for the number of questions
    try:
        num_questions = int(input("Please enter the number of quiz questions to generate: "))
        if num_questions <= 0:
            print("Number of questions must be a positive integer. Defaulting to 1 question.")
            num_questions = 1
    except ValueError:
        print("Invalid input. Defaulting to 1 question.")
        num_questions = 1

    # Generate the specified number of random quiz questions
    generated_quizzes = []

    print(f"\nStarting to generate {num_questions} quiz questions...")

    for i in range(num_questions):
        print(f"\nGenerating question {i+1}/{num_questions}:")
        quiz_info = generate_random_quiz(ndjson_folder, output_folder, i)  # 传递索引i作为文件名

        if quiz_info:
            generated_quizzes.append(quiz_info)

    # Output generation summary
    print("\n========= Quiz Generation Results =========")
    print(f"Successfully generated questions: {len(generated_quizzes)}/{num_questions}")

    if generated_quizzes:
        print("\nQuestion list:")
        for i, quiz in enumerate(generated_quizzes):
            print(f"{i+1}. Image: {os.path.basename(quiz['image_path'])}, Answer: {quiz['answer']}")

        # Generate summary answer file
        summary_file = os.path.join(output_folder, "quiz_summary.txt")
        with open(summary_file, 'w') as f:
            f.write(f"Generation time: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Total questions: {len(generated_quizzes)}\n\n")

            for i, quiz in enumerate(generated_quizzes):
                f.write(f"Question {i+1}:\n")
                f.write(f"  Image: {os.path.basename(quiz['image_path'])}\n")
                f.write(f"  Answer: {quiz['answer']}\n")
                f.write(f"  Word: {quiz['word']}\n")
                f.write(f"  Source file: {quiz['source_file']}\n\n")

        print(f"\nAnswer summary saved to: {summary_file}")


if __name__ == "__main__":
    main()



Starting to generate 5 quiz questions...

Generating question 1/5:
已选择文件: apple.ndjson
已生成测验题目: quiz_images\0.png
答案已保存至: quiz_images\0.txt

Generating question 2/5:
已选择文件: apple.ndjson
已生成测验题目: quiz_images\1.png
答案已保存至: quiz_images\1.txt

Generating question 3/5:
已选择文件: apple.ndjson
已生成测验题目: quiz_images\2.png
答案已保存至: quiz_images\2.txt

Generating question 4/5:
已选择文件: Eiffel Tower.ndjson
已生成测验题目: quiz_images\3.png
答案已保存至: quiz_images\3.txt

Generating question 5/5:
已选择文件: apple.ndjson
已生成测验题目: quiz_images\4.png
答案已保存至: quiz_images\4.txt

========= Quiz Generation Results =========
Successfully generated questions: 5/5

Question list:
1. Image: 0.png, Answer: apple
2. Image: 1.png, Answer: apple
3. Image: 2.png, Answer: apple
4. Image: 3.png, Answer: Eiffel Tower
5. Image: 4.png, Answer: apple

Answer summary saved to: quiz_images\quiz_summary.txt


In [32]:
def add_noise_lines(image_folder, num_lines_range=(2000, 5000), line_length_range=(3, 10), line_width_range=(1, 2)):
    """
    遍历文件夹中的所有PNG图片，在每张图片上添加密集的黑色小短细线作为噪声

    参数:
    image_folder - 包含PNG图片的文件夹路径
    num_lines_range - 每张图片添加的噪声线条数量范围，格式为(最小值, 最大值)
    line_length_range - 噪声线条长度范围，格式为(最小值, 最大值)
    line_width_range - 噪声线条宽度范围，格式为(最小值, 最大值)
    """
    # 获取文件夹中的所有PNG图片
    image_files = glob.glob(os.path.join(image_folder, "*.png"))

    if not image_files:
        print(f"错误：在 {image_folder} 中没有找到PNG图片")
        return

    print(f"\n开始处理图片，添加黑色噪声线条...")

    for image_path in image_files:
        try:
            # 打开图片
            img = Image.open(image_path)
            draw = ImageDraw.Draw(img)

            # 获取图片尺寸
            width, height = img.size

            # 随机决定要添加的噪声线条数量
            num_lines = random.randint(num_lines_range[0], num_lines_range[1])

            for _ in range(num_lines):
                # 随机选择起点
                x1 = random.randint(0, width)
                y1 = random.randint(0, height)

                # 随机选择线条长度和角度
                line_length = random.randint(line_length_range[0], line_length_range[1])
                angle = random.uniform(0, 2 * 3.14159)  # 随机角度（弧度）

                # 计算终点坐标
                x2 = int(x1 + line_length * np.cos(angle))
                y2 = int(y1 + line_length * np.sin(angle))

                # 确保终点在图像范围内
                x2 = max(0, min(width, x2))
                y2 = max(0, min(height, y2))

                # 随机线条宽度
                line_width = random.randint(line_width_range[0], line_width_range[1])

                # 绘制黑色噪声线条
                draw.line([(x1, y1), (x2, y2)], fill="black", width=line_width)

            # 保存修改后的图片
            img.save(image_path)
            print(f"已添加噪声: {os.path.basename(image_path)}")

        except Exception as e:
            print(f"处理 {os.path.basename(image_path)} 时出错: {str(e)}")

    print(f"已完成所有图片的噪声处理，共 {len(image_files)} 张图片")

# 调用函数处理quiz_images文件夹中的图片
output_folder = "quiz_images"  # 您的图片输出文件夹
add_noise_lines(output_folder)



开始处理图片，添加黑色噪声线条...
已添加噪声: 0.png
已添加噪声: 1.png
已添加噪声: 2.png
已添加噪声: 3.png
已添加噪声: 4.png
已完成所有图片的噪声处理，共 5 张图片
